# Question 1 - Chargement et exploration du jeu de donnees

## Objectif

Cette premiere etape a pour but de charger le jeu de donnees
`sfarrukhm/intel-image-classification` depuis Hugging Face, d'en
comprendre la structure et d'en explorer le contenu afin de
preparer les etapes suivantes (construction et entrainement d'un
reseau convolutif).

Les analyses effectuees ici porteront notamment sur :

- le nombre d'images et la repartition entre apprentissage et test ;
- les six classes presentes ;
- la nature des images (format, dimensions, mode couleur) ;
- l'equilibre des classes ;
- les statistiques pixel (moyenne et ecart-type par canal RGB).


## 1.1 Imports et reproductibilite

On importe les bibliotheques necessaires et l'on fixe les graines
aleatoires afin de rendre les experiences reproductibles. Cela
permet de comparer les resultats entre plusieurs executions
successives (tirages aleatoires, initialisation des poids du
reseau, etc.).


In [ ]:
# --- Bibliotheques de base ------------------------------------
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Jeu de donnees (Hugging Face) ---------------------------
from datasets import load_dataset
from huggingface_hub import login

# --- Traitement d'images --------------------------------------
from PIL import Image

# --- Reproductibilite -----------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print("Bibliotheques importees avec succes (SEED = %d)." % SEED)


## 1.2 Chargement du jeu de donnees

Le token Hugging Face est recupere depuis le fichier .env
(jamais ecrit en clair dans le code). Si le token est absent,
le chargement se fait en mode public, plus lent mais fonctionnel.


In [ ]:
# --- Chargement securise du token Hugging Face --------------
# Chemin ABSOLU vers .env pour etre independant du cwd Jupyter.
import os
from pathlib import Path

HF_TOKEN = None
CHEMIN_ENV = Path(r'C:\Users\KOURO\Desktop\Mini projet\.env')
if CHEMIN_ENV.exists():
    for raw_line in CHEMIN_ENV.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == 'HF_TOKEN':
            HF_TOKEN = value.strip().strip(chr(34)).strip(chr(39))
            break

print('CWD Jupyter :', os.getcwd())
print('.env trouve  :', CHEMIN_ENV.exists())
print('Token charge :', 'OUI' if HF_TOKEN else 'NON',
      '(longueur=%d)' % len(HF_TOKEN) if HF_TOKEN else '')

# --- Authentification et chargement du dataset --------------
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    dataset = load_dataset(
        'sfarrukhm/intel-image-classification',
        token=HF_TOKEN,
    )
else:
    print('HF_TOKEN absent : chargement public (plus lent).')
    dataset = load_dataset('sfarrukhm/intel-image-classification')

print(dataset)


### Interpretation

Le jeu de donnees est un objet DatasetDict contenant deux
sous-ensembles :

- train : images d'apprentissage, utilisees pour construire le modele ;
- test : images de test, utilisees pour evaluer la generalisation du
  modele sur des exemples jamais vus.

La totalite des 17 034 images est deja decoupee par le createur du
dataset, ce qui evite de devoir realiser nous-memes le
train_test_split (meme si une partie de train sera ensuite isolee
en validation).


## 1.3 Taille des sous-ensembles

On verifie explicitement le nombre d'images par split afin de
s'assurer que les valeurs correspondent a la description du
dataset (14 034 images d'apprentissage et 3 000 images de test).


In [ ]:
for split in dataset.keys():
    print('{0:>5} : {1:>5} images'.format(split, len(dataset[split])))

total = sum(len(dataset[s]) for s in dataset.keys())
print('Total : %d images.' % total)


### Interpretation

La repartition est d'environ 82,4 % pour l'apprentissage et
17,6 % pour le test. Cette proportion est coherente avec un
partage classique (~80/20) et garantit un volume suffisant
d'images testees pour une evaluation fiable.


## 1.4 Structure d'un exemple

Chaque ligne du dataset est un dictionnaire avec deux champs :
image (l'image PIL) et label (entier designant la classe).
On observe egalement les proprietes detaillees de l'image.


In [ ]:
exemple = dataset['train'][0]
print('Cles d\'un exemple :', list(exemple.keys()))

print('Features du split train :')
print(dataset['train'].features)

print('Proprietes de l\'image exemple :')
print('  type       :', type(exemple['image']).__name__)
print('  taille     :', exemple['image'].size, '(largeur, hauteur)')
print('  mode       :', exemple['image'].mode)
print('  label      :', exemple['label'])


### Interpretation

L'image est un objet PIL.Image en mode RGB (3 canaux). La
taille precise sera etudiee sur l'ensemble du corpus un peu plus
loin (les dimensions ne sont pas strictement identiques pour
toutes les images, ce qui impose un redimensionnement avant
l'entrainement).


## 1.5 Classes du probleme

On extrait la liste officielle des classes via les metadonnees
du dataset, ce qui garantit la coherence avec les labels
numeriques.


In [ ]:
labels = dataset['train'].features['label'].names
print('Classes :', labels)
print('Nombre de classes :', len(labels))
for i, nom in enumerate(labels):
    print('  %d -> %s' % (i, nom))


### Interpretation

Le probleme est une classification supervisee multi-classes a 6
categories, avec un label entier unique par image. On remarque
que certaines classes sont visuellement proches (par exemple
mountain / glacier, ou buildings / street), ce qui constituera
un defi pour le modele.


## 1.6 Visualisation d'un exemple par classe

Pour mieux apprehender la nature des images, on tire aleatoirement
une image par classe et on les affiche dans une grille 2x3.


In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for classe_id, ax in enumerate(axes.flat):
    indices = [i for i, lab in enumerate(dataset['train']['label'])
                if lab == classe_id]
    idx = int(rng.choice(indices))
    ex = dataset['train'][idx]
    ax.imshow(ex['image'])
    ax.set_title('{0} (label={1})'.format(labels[classe_id], classe_id), fontsize=12)
    ax.axis('off')
plt.suptitle('Echantillons : une image par classe', fontsize=14)
plt.tight_layout()
plt.show()


### Interpretation

L'observation visuelle confirme la diversite des scenes et
l'eventuelle ambiguite entre certaines classes. Par exemple,
une image peut contenir a la fois des immeubles et une rue
(buildings vs street), ou encore des montagnes et un glacier
en arriere-plan (mountain vs glacier). Cette proximite visuelle
sera un facteur determinant de la performance du futur
classificateur.


## 1.7 Dimensions des images

Au lieu de parcourir les 14 034 images (tres couteux en temps),
on realise un sondage aleatoire de 500 images. La largeur est
toujours 150 px, mais la hauteur varie selon les images.


In [ ]:
rng = np.random.default_rng(SEED)
TAILLE_SONDAGE = 500
indices_sondage = rng.choice(len(dataset['train']),
                             size=TAILLE_SONDAGE, replace=False)

largeurs, hauteurs = [], []
for i in indices_sondage:
    w, h = dataset['train'][int(i)]['image'].size
    largeurs.append(w)
    hauteurs.append(h)

print('Largeurs uniques  :', sorted(set(largeurs)))
print('Hauteurs uniques  :', sorted(set(hauteurs)))
print('Total sondage     :', TAILLE_SONDAGE, 'images')


### Interpretation

Toutes les images ont une largeur fixe de 150 px ; en revanche,
elles presentent differentes hauteurs. Cette heterogeneite nous
oblige a appliquer un redimensionnement (resize) avant l'entree
dans le reseau de neurones, qui exige des dimensions constantes
au sein d'un batch.


## 1.8 Modes couleur

On verifie egalement que toutes les images sont bien en mode
RGB (3 canaux) et non en niveaux de gris ou en mode palette.


In [ ]:
TAILLE_SONDAGE_MODES = 500
indices_sondage_modes = rng.choice(len(dataset['train']),
                                    size=TAILLE_SONDAGE_MODES, replace=False)
modes = [dataset['train'][int(i)]['image'].mode
         for i in indices_sondage_modes]
print('Modes uniques trouves :', set(modes))
print('Total sondage :', TAILLE_SONDAGE_MODES, 'images')


### Interpretation

Toutes les images sondage sont en mode RGB. Aucune conversion
speciale (par exemple grayscale -> RGB) ne sera donc necessaire :
on peut directement empiler les images en tenseurs a 3 canaux.


## 1.9 Distribution des classes (jeu d'apprentissage)

On denombre les images par classe et l'on calcule le pourcentage
de chaque classe, afin de detecter un eventuel desequilibre.


In [ ]:
labels_train = dataset['train']['label']
distribution = pd.Series(labels_train).value_counts().sort_index()

distribution_df = pd.DataFrame({
    'Classe': labels,
    'Nombre d\'images': distribution.values,
})
distribution_df['Pourcentage (%)'] = (
    distribution_df['Nombre d\'images']
    / distribution_df['Nombre d\'images'].sum() * 100
).round(2)
distribution_df


### Visualisation de la distribution

Le tableau precedent est complete par un diagramme en barres qui
facilite la comparaison visuelle entre les classes.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(distribution_df['Classe'],
              distribution_df['Nombre d\'images'],
              color=sns.color_palette('pastel'))
for bar, val in zip(bars, distribution_df['Nombre d\'images']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20, str(val),
            ha='center', va='bottom', fontsize=10)
ax.set_title('Nombre d\'images par classe (jeu d\'apprentissage)')
ax.set_xlabel('Classe'); ax.set_ylabel('Nombre d\'images')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()


### Interpretation

Les classes presentent des effectifs compris entre 2 191 et
2 512 images, soit un ecart relatif d'environ 14 %. Cette
distribution est relativement equilibree : le ratio entre la
classe la plus representee et la moins representee est proche
de 1. Une analyse plus fine (avec calcul de coefficients de
variation et eventuels poids de classes) sera menee en
Question 2.


## 1.10 Statistiques pixel par canal

Pour evaluer la necessite d'une normalisation particuliere (par
exemple centrage-reduction pour un reseau pre-entraine), on
calcule la moyenne et l'ecart-type des canaux R, G, B sur un
echantillon aleatoire d'images (sondage = 1 000 images pour
limiter le cout memoire).


In [ ]:
rng = np.random.default_rng(SEED)
TAILLE_SONDAGE_PIXELS = 1000
indices_pixels = rng.choice(len(dataset['train']),
                             size=TAILLE_SONDAGE_PIXELS, replace=False)

somme = np.zeros(3)
somme_carres = np.zeros(3)
nb_pixels_total = 0
for i in indices_pixels:
    arr = np.asarray(dataset['train'][int(i)]['image'], dtype=np.float64) / 255.0
    pixels = arr.reshape(-1, 3)
    somme        += pixels.sum(axis=0)
    somme_carres += (pixels ** 2).sum(axis=0)
    nb_pixels_total += pixels.shape[0]

moyennes = somme / nb_pixels_total
variances = np.maximum(somme_carres / nb_pixels_total - moyennes ** 2, 0.0)
ecarts_types = np.sqrt(variances)

stats_df = pd.DataFrame({
    'Canal': ['Rouge (R)', 'Vert (G)', 'Bleu (B)'],
    'Moyenne': moyennes.round(4),
    'Ecart-type': ecarts_types.round(4),
})
print(stats_df.to_string(index=False))


### Interpretation

Les moyennes par canal sont tres proches (R ~0.43, G ~0.46,
B ~0.46), ce qui traduit un dataset sans teinte dominante :
coherent avec la diversite des scenes (ciel, vegetation,
beton...). Les ecarts-types sont egalement voisins (~0.27),
avec un canal B legerement plus disperse (~0.30) en raison
des variations bleutees (mer, glaciers, ciel).

Ces valeurs seront utilisees pour la normalisation :

- pour le CNN cree de toutes pieces (Question 3), un simple
  rescaling x / 255 suffit ;
- pour le modele pre-entraine ResNet (Question 9), on appliquera
  la centrage-reduction d'ImageNet (moyennes
  [0.485, 0.456, 0.406], ecarts-types [0.229, 0.224, 0.225]).


## 1.11 Distribution des intensites pixel

Pour completer l'analyse statistique, on visualise la
distribution des intensites pixel par canal sur le meme
echantillon.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
couleurs = [('Rouge (R)', 'red'), ('Vert (G)', 'green'), ('Bleu (B)', 'blue')]
for canal_idx, (nom, couleur) in enumerate(couleurs):
    intensites = []
    for i in indices_pixels:
        arr = np.asarray(dataset['train'][int(i)]['image'])[:, :, canal_idx].ravel()
        intensites.append(arr)
    intensites = np.concatenate(intensites)
    axes[canal_idx].hist(intensites, bins=50, color=couleur, alpha=0.7)
    axes[canal_idx].set_title(nom)
    axes[canal_idx].set_xlabel('Intensite (0-255)')
    axes[canal_idx].set_ylabel('Nombre de pixels')
plt.suptitle('Distribution des intensites pixel par canal')
plt.tight_layout(); plt.show()


### Interpretation

Les trois histogrammes presentent une distribution etalee,
avec une legere concentration vers les valeurs moyennes
(entre 80 et 160). Aucune saturation importante n'est visible
aux extremites (0 ou 255), ce qui indique que le dataset
contient peu d'images entierement sombres ou entierement
claires. La forme generale est proche d'une gaussienne, ce qui
est benefique pour l'entrainement des reseaux convolutifs.


## Conclusion de la Question 1

Le jeu de donnees sfarrukhm/intel-image-classification contient
**17 034 images** reparties en **14 034 images d'apprentissage**
et **3 000 images de test**. Il s'agit d'un probleme de
**classification supervisee a 6 classes** : buildings, forest,
glacier, mountain, sea, street.

Les images sont en mode **RGB** et ont une largeur de **150 px**,
avec une hauteur variable. Une etape de **redimensionnement**
sera donc necessaire avant l'entrainement.

Les statistiques pixel montrent une distribution equilibree des
canaux RGB, sans teinte dominante : un simple rescaling x / 255
suffit pour le CNN from-scratch, tandis que la normalisation
ImageNet sera utilisee pour ResNet.

La distribution des classes est **relativement equilibree**
(ecart d'environ 14 % entre la classe la plus et la moins
representee). Une analyse plus fine sera menee en Question 2
pour determiner si un reequilibrage est malgre tout necessaire.


### 2 - Ce dataset est-il équilibré ? Est-il nécessaire de rééquilibrer les données ? Le faire si besoin est.

Étape 1:Quantifier l'équilibre des classes

In [ ]:
# Récupération des noms des classes
labels = dataset["train"].features["label"].names

# Récupération des labels du jeu d'entraînement
train_labels = dataset["train"]["label"]

# Comptage des images par classe
distribution = pd.Series(train_labels).value_counts().sort_index()

# Création du tableau
distribution_df = pd.DataFrame({
    "Classe": labels,
    "Nombre d'images": distribution.values
})

# Calcul du pourcentage
distribution_df["Pourcentage"] = (
    distribution_df["Nombre d'images"]
    / distribution_df["Nombre d'images"].sum()
    * 100
)

distribution_df

,Classe,Nombre d'images,Pourcentage
0,buildings,2191,15.612085
1,forest,2271,16.182129
2,glacier,2404,17.129828
3,mountain,2512,17.899387
4,sea,2274,16.203506
5,street,2382,16.973065


### Étape 2 — Application d'une pondération des classes (mesure défensive)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Récupère les labels et les noms de classes depuis le dataset déjà chargé
labels = dataset["train"].features["label"].names
NUM_CLASSES = len(labels)

# Compte les images par classe
labels_train = np.array(dataset["train"]["label"])
decomptes_entrainement = np.bincount(labels_train, minlength=NUM_CLASSES)

print("=== Décompte par classe (train) ===")
for i, c in enumerate(labels):
    print(f"  {c:<12} : {decomptes_entrainement[i]}")

# Mesure du déséquilibre
ratio = decomptes_entrainement.max() / decomptes_entrainement.min()
cv = decomptes_entrainement.std() / decomptes_entrainement.mean()
print(f"\nRatio max/min : {ratio:.3f}")
print(f"CV (std/mean) : {cv:.3f}")

# Poids de classes (méthode sklearn "balanced")
poids_bruts = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=labels_train,
)
dictionnaire_poids = {i: float(w) for i, w in enumerate(poids_bruts)}

print("\n=== Poids de classes (balanced) ===")
for i, c in enumerate(labels):
    print(f"  {c:<12} : w = {dictionnaire_poids[i]:.4f}")
print(f"\nDictionnaire : {dictionnaire_poids}")

=== Décompte par classe (train) ===
  buildings    : 2191
  forest       : 2271
  glacier      : 2404
  mountain     : 2512
  sea          : 2274
  street       : 2382

Ratio max/min : 1.147
CV (std/mean) : 0.045

=== Poids de classes (balanced) ===
  buildings    : w = 1.0675
  forest       : w = 1.0299
  glacier      : w = 0.9730
  mountain     : w = 0.9311
  sea          : w = 1.0286
  street       : w = 0.9819

Dictionnaire : {0: 1.067549064354176, 1: 1.0299427564949362, 2: 0.9729617304492513, 3: 0.9311305732484076, 4: 1.0285839929639402, 5: 0.9819479429051218}


Étape 3: Vérifier également le jeu de test

In [ ]:
# Distribution des classes dans le jeu de test
test_labels = dataset["test"]["label"]

distribution_test = pd.Series(test_labels).value_counts().sort_index()

distribution_test_df = pd.DataFrame({
    "Classe": labels,
    "Nombre d'images": distribution_test.values
})

distribution_test_df["Pourcentage"] = (
    distribution_test_df["Nombre d'images"]
    / distribution_test_df["Nombre d'images"].sum() * 100
)

distribution_test_df

,Classe,Nombre d'images,Pourcentage
0,buildings,437,14.566667
1,forest,474,15.800000
2,glacier,553,18.433333
3,mountain,525,17.500000
4,sea,510,17.000000
5,street,501,16.700000


## Conclusion — Question 2

La distribution des classes est la suivante :
- **Entraînement** : buildings : 2 191, forest : 2 271, glacier : 2 404,
 mountain : 2 512, sea : 2 274, street : 2 382.
- **Test**         : buildings :  437, forest :   474, glacier :   553,
 mountain :   525, sea :   510, street :   501.

Les métriques d'équilibre sont :

| Jeu | Ratio min/max | Coef. de variation |
|---|---|---|
| Entraînement | 0,872 | 0,045 |
| Test         | 0,790 | 0,077 |

Les deux ratios sont supérieurs au seuil empirique de 0,8 (entraînement)
ou très proches (test). Le dataset est par conséquent considéré comme
**équilibré**.

Aucun rééquilibrage par sur/sous‑échantillonnage n'est nécessaire :
ces opérations présenteraient ici plus d'inconvénients (perte
d'information ou risque de sur‑apprentissage) que d'avantages.

Les poids calculés restent très proches de 1 (extrêmes : 0,929 pour
`mountain` et 1,065 pour `buildings`). L'effet de la pondération sur la
loss sera donc marginal : il corrige le léger déséquilibre observé,
sans modifier les données ni introduire de sur‑apprentissage. On la
conserve comme bonne pratique défensive.

**Poids retenus** :
`{0: 1.065, 1: 1.028, 2: 0.971, 3: 0.929, 4: 1.027, 5: 0.980}`.
Ces poids seront passés à `model.fit()` à partir de la question 4.

### 3.Construire un réseau de neurones convolutif pour résoudre ce problème de classification. Il devra contenir au minimum les éléments suivants : couches de convolution, couche de "pooling", "dropout", couches cachées complètement connectées. Vous êtes libres d'ajouter d'autres éléments.

In [ ]:
!pip install tensorflow

Etape 1: Pipeline de données

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# Pipeline PyTorch : chargement a la volee pour eviter de stocker
# toutes les images en RAM.
TAILLE_IMAGE = (96, 96)
TAILLE_LOT   = 64 if torch.cuda.is_available() else 32
# num_workers=0 : evite les crashes multiprocessing sous
# Python 3.14 / Windows (le pickling HF datasets est sensible).
NB_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
labels = dataset['train'].features['label'].names

class IntelSceneDataset(Dataset):
    def __init__(self, hf_split, indices, transform):
        self.hf_split  = hf_split
        self.indices   = np.asarray(indices, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        ex = self.hf_split[int(self.indices[idx])]
        image = ex['image'].convert('RGB')
        label = int(ex['label'])
        image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize(TAILLE_IMAGE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize(TAILLE_IMAGE),
    transforms.ToTensor(),
])

labels_train_all = np.array(dataset['train']['label'], dtype=np.int64)
indices_all = np.arange(len(labels_train_all))

indices_train, indices_val = train_test_split(
    indices_all, test_size=0.20, random_state=SEED,
    stratify=labels_train_all, shuffle=True,
)

train_ds = IntelSceneDataset(dataset['train'], indices_train, train_transform)
val_ds   = IntelSceneDataset(dataset['train'], indices_val,   eval_transform)
test_ds  = IntelSceneDataset(dataset['test'],  np.arange(len(dataset['test'])), eval_transform)

donnees_entrainement = DataLoader(
    train_ds, batch_size=TAILLE_LOT, shuffle=True,
    num_workers=NB_WORKERS, pin_memory=PIN_MEMORY,
)
donnees_validation = DataLoader(
    val_ds, batch_size=TAILLE_LOT, shuffle=False,
    num_workers=NB_WORKERS, pin_memory=PIN_MEMORY,
)
donnees_test = DataLoader(
    test_ds, batch_size=TAILLE_LOT, shuffle=False,
    num_workers=NB_WORKERS, pin_memory=PIN_MEMORY,
)

print('Device CUDA dispo:', torch.cuda.is_available())
print('Train size:', len(train_ds), '| Val size:', len(val_ds),
      '| Test size:', len(test_ds))
print('Batch size:', TAILLE_LOT, '| Workers:', NB_WORKERS)


étape 2: l'architecture du CNN 

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilise:", device)

class CNNLight500k(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(3, 48, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 48, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.15),

            # Bloc 2
            nn.Conv2d(48, 96, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 96, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.20),

            # Bloc 3
            nn.Conv2d(96, 160, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(160),
            nn.ReLU(inplace=True),
            nn.Conv2d(160, 160, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(160),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(160, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes),
)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

modele_torch = CNNLight500k(num_classes=len(labels)).to(device)

def compter_parametres(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params entrainables: {compter_parametres(modele_torch):,}")

Model: "CNN_from_scratch_gap"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aug_flip (RandomFlip)           │ (None, 112, 112, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aug_rot (RandomRotation)        │ (None, 112, 112, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aug_zoom (RandomZoom)           │ (None, 112, 112, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aug_shift (RandomTranslation)   │ (None, 112, 112, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aug_contrast (RandomContrast)   │ (None, 112, 112, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 112, 112, 64)   │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_1 (BatchNormalization)      │ (None, 112, 112, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1_1 (Activation)            │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 112, 112, 64)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 112, 112, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1_2 (Activation)            │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sdrop1 (SpatialDropout2D)       │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_1 (BatchNormalization)      │ (None, 56, 56, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2_1 (Activation)            │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 56, 56, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 56, 56, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2_2 (Activation)            │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sdrop2 (SpatialDropout2D)       │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 28, 28, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_1 (BatchNormalization)      │ (None, 28, 28, 256)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3_1 (Activation)            │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_2 (Conv2D)                │ (None, 28, 28, 256)    │       589,82

 Total params: 1,216,550 (4.64 MB)

 Trainable params: 1,214,758 (4.63 MB)

 Non-trainable params: 1,792 (7.00 KB)

Total params: 1,216,550


### Analyse — Question 3

Le point critique du modèle précédent était la transition `Flatten -> Dense`, qui concentrait l'essentiel des paramètres et favorisait le surapprentissage.

Le modèle a donc été corrigé avec une tête plus adaptée :
- `GlobalAveragePooling2D` au lieu de `Flatten`,
- deux couches denses modérées (`Dense(192)` puis `Dense(96)`),
- régularisation L2 et dropout pour limiter le surapprentissage.

Architecture retenue :
- 3 blocs convolutionnels (64 -> 128 -> 256 filtres),
- `BatchNormalization` dans chaque bloc,
- `SpatialDropout2D` après pooling,
- augmentation d'images (`flip`, `rotation`, `zoom`, `translation`, `contrast`).

Ce choix réduit fortement le nombre total de paramètres, accélère l'entraînement et améliore généralement la généralisation sur un dataset de taille moyenne comme Intel (6 classes).

## 4. Entrainement du modele et mesure de la performance

L'objectif est d'entrainer le CNN defini en Question 3 sur le
jeu d'entrainement, de suivre la convergence sur le jeu de
validation, puis d'evaluer la performance finale sur le jeu
de test (jamais vu pendant l'entrainement).

**Plan :**

- configuration de l'optimiseur, du scheduler et de la loss ;
- boucle d'entrainement avec early stopping et sauvegarde du
  meilleur modele ;
- evaluation finale sur le test (accuracy, loss, matrice de
  confusion, rapport par classe) ;
- visualisation des courbes d'apprentissage.


In [ ]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

# --- Repertoire de checkpoints -----------------------------
DOSSIER_Q4 = 'checkpoints_q4_torch'
DERNIER_CKPT   = os.path.join(DOSSIER_Q4, 'dernier_modele.pt')
MEILLEUR_CKPT  = os.path.join(DOSSIER_Q4, 'meilleur_modele.pt')
FICHIER_LOG    = os.path.join(DOSSIER_Q4, 'historique_q4.csv')
os.makedirs(DOSSIER_Q4, exist_ok=True)

# --- Loss : CrossEntropy avec poids de classes (Q2) -------
poids_classes = None
if 'dictionnaire_poids' in globals():
    poids_classes = torch.tensor(
        [dictionnaire_poids[i] for i in range(len(labels))],
        dtype=torch.float32, device=device,
    )
criterion = nn.CrossEntropyLoss(weight=poids_classes)

# --- Optimiseur AdamW + scheduler ReduceLROnPlateau ------
optimizer = optim.AdamW(modele_torch.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6,
)

# --- Mixed precision (FP16) - accelere sur GPU ------------
USE_AMP = (device.type == 'cuda')
scaler = GradScaler('cuda', enabled=USE_AMP)


def train_one_epoch(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total   += targets.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total   += targets.size(0)
    return running_loss / total, correct / total

print(f'Configuration prete - device={device}, AMP={USE_AMP}')


In [ ]:
# --- Boucle d'entrainement --------------------------------
NOMBRE_EPOQUES = 15
PATIENCE = 3      # early stopping
best_val_loss = float('inf')
best_val_acc  = 0.0
wait = 0
logs = []

# Historique exploitable par la cellule de plot
class HistoryLike: pass
historique = HistoryLike()
historique.history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

print('Debut entrainement PyTorch...')
for epoch in range(1, NOMBRE_EPOQUES + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(modele_torch, donnees_entrainement)
    val_loss,   val_acc   = eval_one_epoch(modele_torch, donnees_validation)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    for key, val in zip(
        ('loss', 'accuracy', 'val_loss', 'val_accuracy'),
        (train_loss, train_acc, val_loss, val_acc),
    ):
        historique.history[key].append(val)

    logs.append({
        'epoch': epoch, 'loss': train_loss, 'accuracy': train_acc,
        'val_loss': val_loss, 'val_accuracy': val_acc, 'learning_rate': lr,
    })
    pd.DataFrame(logs).to_csv(FICHIER_LOG, index=False)

    # Sauvegarde du dernier et du meilleur
    torch.save({
        'epoch': epoch, 'model_state_dict': modele_torch.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss, 'val_accuracy': val_acc,
    }, DERNIER_CKPT)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(modele_torch.state_dict(), MEILLEUR_CKPT)

    # Early stopping sur val_loss
    if val_loss < best_val_loss - 1e-4:
        best_val_loss, wait = val_loss, 0
    else:
        wait += 1

    dt = time.time() - t0
    print(
        f'Epoch {epoch:02d}/{NOMBRE_EPOQUES} | '
        f'loss={train_loss:.4f} acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | '
        f'lr={lr:.2e} | {dt:.1f}s'
    )
    if wait >= PATIENCE:
        print('EarlyStopping active (val_loss ne s\'ameliore plus).')
        break

# Recharge le meilleur modele pour l'evaluation finale
modele_torch.load_state_dict(torch.load(MEILLEUR_CKPT, map_location=device))
print(f'\nMeilleur modele recharge - val_accuracy={best_val_acc:.4f}')


In [ ]:
# --- Evaluation finale sur le jeu de test ------------------
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def eval_test_complet(model, loader):
    model.eval()
    tous_pred, tous_true = [], []
    running_loss, correct, total = 0.0, 0, 0
    criterion_test = nn.CrossEntropyLoss(weight=poids_classes)
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion_test(logits, targets)
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        tous_pred.extend(preds.cpu().numpy())
        tous_true.extend(targets.cpu().numpy())
        correct += (preds == targets).sum().item()
        total   += targets.size(0)
    return (running_loss/total, correct/total,
            np.array(tous_pred), np.array(tous_true))

perte_test, exact_test, y_pred, y_true = eval_test_complet(
    modele_torch, donnees_test,
)
print(f'Loss test : {perte_test:.4f}')
print(f'Accuracy test : {exact_test:.4f} ({exact_test*100:.2f}%)')


In [ ]:
# --- Rapport par classe + matrice de confusion ------------
print('Rapport de classification (jeu de test) :\n')
print(classification_report(
    y_true, y_pred,
    target_names=labels, digits=4,
))

mat_conf = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    mat_conf, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax,
)
ax.set_xlabel('Predictions'); ax.set_ylabel('Vraies classes')
ax.set_title('Matrice de confusion - CNN from-scratch (test)')
plt.xticks(rotation=30); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
# --- Courbes d'apprentissage (loss + accuracy) ------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historique.history['loss'], label='train')
ax1.plot(historique.history['val_loss'], label='validation')
ax1.set_title('Evolution de la loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True)

ax2.plot(historique.history['accuracy'], label='train')
ax2.plot(historique.history['val_accuracy'], label='validation')
ax2.set_title('Evolution de l\'accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()


## Conclusion - Question 4

L'entrainement suit un protocole rigoureux : **train** pour
l'apprentissage, **validation** pour le suivi et l'early
stopping, **test** pour l'evaluation finale uniquement.

**Optimiseur** : AdamW (`lr=3e-4`, `weight_decay=1e-4`) avec
scheduler `ReduceLROnPlateau` (facteur 0.5, patience 2) ;
**regularisation** : CrossEntropyLoss ponderee par les poids
de classes (Q2), Dropout2D dans les blocs convolutifs, AMP
FP16 sur GPU pour accelerer les calculs.

Les courbes d'apprentissage permettent de verifier la
convergence et l'absence de surapprentissage : si l'ecart
train/validation reste faible (<5 points d'accuracy), le
modele generalise bien. Les metriques finales (accuracy
test, F, Fclass) seront reportees apres execution et
comparees a celles des questions suivantes (GridSearchCV
Q5, augmentation Q6, ResNet Q9).


### 5. Faire une recherche de meilleurs hyperparamètres avec la fonction "GridSearchCV“.

Étape 1 — Vérifications préalables


In [ ]:
# Vérifier scikeras (wrapper sklearn pour Keras)
try:
    from scikeras.wrappers import KerasClassifier
    print("scikeras OK")
except ImportError:
    import subprocess
    print("Installation de scikeras…")
    subprocess.run(["pip", "install", "scikeras"], check=True)
    from scikeras.wrappers import KerasClassifier
    print("scikeras installé OK")

# Vérifier sklearn
import sklearn
print(f"scikit-learn : {sklearn.__version__}")

scikeras OK
scikit-learn : 1.9.1


Étape 2 — Fonction de construction du modèle

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def construire_modele(dense_units=256, learning_rate=1e-3):
    """Construit le CNN avec deux hyperparamètres variables."""
    modele = keras.Sequential([
        layers.Input(shape=(150, 150, 3)),        # Bloc 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        # Tête dense
        layers.Flatten(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(6, activation="softmax"),
    ])
    modele.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modele

Étape 3 — Wrapper sklearn et grille d'hyperparamètres

In [ ]:
classifieur = KerasClassifier(
    model=construire_modele,
    epochs=2,                # plus léger pour éviter les dépassements mémoire
    batch_size=24,
    verbose=0,
)

grille = {
    "model__dense_units"   : [128, 256],
    "model__learning_rate" : [1e-3, 1e-4],
}

recherche = GridSearchCV(
    estimator=classifieur,
    param_grid=grille,
    cv=2,                    # réduit pour limiter la mémoire
    scoring="accuracy",
    n_jobs=1,                # pas de parallélisme en CPU/RAM limitée
    verbose=2,
)

Étape 4 — Lancement de la recherche

In [ ]:
# Sauvegarde des poids du modèle précédent pour pouvoir y revenir
modele_cnn.save_weights("poids_cnn_avant_gridsearch.weights.h5")
print("Poids du modèle Q4 sauvegardés.")

import numpy as np

if "images_train" not in globals() or "labels_train" not in globals():
    raise RuntimeError("images_train/labels_train absents de la mémoire. Relance la cellule de conversion des données avant le GridSearch.")

# Sous-échantillonnage pour éviter le MemoryError
rng = np.random.default_rng(SEED)
taille = len(images_train)
fraction = 0.25
n_echantillon = int(taille * fraction)
indices = rng.choice(taille, size=n_echantillon, replace=False)

images_train_gs = images_train[indices]
labels_train_gs = labels_train[indices]

print("\nLancement du GridSearchCV en mode mémoire réduite…")
print(f"  Taille totale train : {taille}")
print(f"  Taille sous-échantillon : {n_echantillon}")
print(f"  Combinaisons : {len(grille['model__dense_units']) * len(grille['model__learning_rate'])}")
print("  Folds CV     : 2")
print("  Epochs/fit   : 2")
print(f"  Total fits   : {2 * 2 * 2} entraînements\n")

recherche.fit(images_train_gs, labels_train_gs)

print("\n=== Résultats ===")
print("Meilleurs paramètres :", recherche.best_params_)
print("Meilleur accuracy CV  :", recherche.best_score_)

# Libère la RAM immédiatement après la recherche
del images_train_gs, labels_train_gs, indices

Poids du modèle Q4 sauvegardés.

Lancement du GridSearchCV (≈ 2h sur CPU)…
  Combinaisons : 4
  Folds CV     : 3
  Epochs/fit   : 3
  Total fits : 12 = 12 entraînements

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END .model__dense_units=128, model__learning_rate=0.001; total time=   8.3s
[CV] END .model__dense_units=128, model__learning_rate=0.001; total time=   3.3s
[CV] END .model__dense_units=128, model__learning_rate=0.001; total time=   2.9s
[CV] END model__dense_units=128, model__learning_rate=0.0001; total time=   3.2s
[CV] END model__dense_units=128, model__learning_rate=0.0001; total time=   3.2s
[CV] END model__dense_units=128, model__learning_rate=0.0001; total time=   2.6s
[CV] END .model__dense_units=256, model__learning_rate=0.001; total time=   3.0s
[CV] END .model__dense_units=256, model__learning_rate=0.001; total time=   3.5s
[CV] END .model__dense_units=256, model__learning_rate=0.001; total time=   2.9s


MemoryError: Unable to allocate 672. MiB for an array with shape (4678, 112, 112, 3) and data type float32

In [ ]:
modes = set(image.mode for image in dataset["train"]["image"])

print("Modes d'image :", modes)

Modes d'image : {'RGB'}


In [ ]:
resultats = pd.DataFrame(recherche.cv_results_)
resultats = resultats[["param_model__dense_units",
                        "param_model__learning_rate",
                        "mean_test_score", "std_test_score",
                        "rank_test_score"]]
resultats = resultats.sort_values("rank_test_score")
print(resultats.to_string(index=False))

 param_model__dense_units  param_model__learning_rate  mean_test_score  std_test_score  rank_test_score
                      256                      0.0001         0.014536        0.011022                1
                      128                      0.0010         0.013111        0.016041                2
                      128                      0.0001         0.012755        0.015539                3
                      256                      0.0010         0.009833        0.011295                4


In [ ]:

## Cellule 2 — Imports et wrapper amélioré

import time
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

def construire_modele(dense_units=256, learning_rate=1e-3, dropout_dense=0.5):
    """CNN avec hyperparamètres variables pour le GridSearchCV."""
    modele = keras.Sequential([
        layers.Input(shape=(150, 150, 3)),        # Bloc 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        # Tête dense
        layers.Flatten(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dropout(dropout_dense),
        layers.Dense(6, activation="softmax"),
    ])
    modele.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modele

classifieur = KerasClassifier(
    model=construire_modele,
    epochs=5,                # ↑ par rapport à 3
    batch_size=32,
    verbose=0,
    # class_weight via fit_kwargs (voir cellule suivante)
)

In [ ]:
grille = {
    "model__dense_units"   : [128, 256],
    "model__learning_rate" : [1e-3, 1e-4],
}

recherche = GridSearchCV(
    estimator=classifieur,
    param_grid=grille,
    cv=2,                    # ↓ par rapport à 3
    scoring="accuracy",
    n_jobs=1,
    verbose=2,
    refit=True,              # ré‑entraîne le meilleur modèle sur tout le train
)

print("Lancement du GridSearchCV (Option A)…")
print(f"  Combinaisons : 4")
print(f"  Folds CV     : 2")
print(f"  Epochs/fit   : 5")
print(f"  Total fits   : 4 × 2 = 8 entraînements\n")

debut = time.time()
recherche.fit(images_train, labels_train)
duree_min = (time.time() - debut) / 60
print(f"\nDurée totale : {duree_min:.1f} min")

print("\n=== Résultats ===")
print("Meilleurs paramètres :", recherche.best_params_)
print("Meilleur accuracy CV  :", recherche.best_score_)

Lancement du GridSearchCV (Option A)…
  Combinaisons : 4
  Folds CV     : 2
  Epochs/fit   : 5
  Total fits   : 4 × 2 = 8 entraînements

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END .model__dense_units=128, model__learning_rate=0.001; total time=24.5min
[CV] END .model__dense_units=128, model__learning_rate=0.001; total time=22.3min
[CV] END model__dense_units=128, model__learning_rate=0.0001; total time=20.2min
[CV] END model__dense_units=128, model__learning_rate=0.0001; total time=20.0min
[CV] END .model__dense_units=256, model__learning_rate=0.001; total time=20.1min
[CV] END .model__dense_units=256, model__learning_rate=0.001; total time=19.8min
[CV] END model__dense_units=256, model__learning_rate=0.0001; total time=20.1min
[CV] END model__dense_units=256, model__learning_rate=0.0001; total time=19.8min

Durée totale : 206.9 min

=== Résultats ===
Meilleurs paramètres : {'model__dense_units': 256, 'model__learning_rate': 0.001}
Meilleur accuracy CV  : 0.011

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[]


In [ ]:
import psutil
vm = psutil.virtual_memory()
print(f"RAM utilisée: {(1-vm.available/vm.total)*100:.1f}% | dispo: {vm.available/1e9:.2f} GB")

RAM utilisée: 88.4% | dispo: 3.96 GB


In [ ]:
import tensorflow as tf, sys, platform
print("Python:", sys.version)
print("Exe:", sys.executable)
print("OS:", platform.platform())
print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs physiques:", tf.config.list_physical_devices("GPU"))
print("GPUs logiques:", tf.config.list_logical_devices("GPU"))

Python: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
Exe: c:\Users\KOURO\anaconda3\envs\deep5\python.exe
OS: Windows-11-10.0.26200-SP0
TF version: 2.21.0
Built with CUDA: False
GPUs physiques: []
GPUs logiques: []


In [ ]:
import torch
print(torch.cuda.is_available())   # doit afficher True
print(torch.cuda.get_device_name(0))  # "NVIDIA GeForce RTX 5060"

RAM utilisée: 74.5% | dispo: 8.67 GB


## 9. Modele pre-entraine (ResNet18)
Chargement d'un modele pre-entraine, personnalisation (couches convolutionnelles + fully connected), entrainement et comparaison avec le meilleur modele de la Question 4.


## 10. Bonus CAM - Class Activation Mapping

### But de la methode
CAM (Class Activation Mapping) sert a **expliquer une prediction** d'un reseau de classification d'images en localisant les regions qui ont le plus contribue a la classe predite.

### Principe de fonctionnement
1. Le reseau doit se terminer par une **derniere couche convolutionnelle**, suivie d'un **Global Average Pooling (GAP)**, puis d'une couche lineaire de classification.
2. Pour une classe c, la couche lineaire fournit des poids w_{k,c} associes a chaque canal k de la carte de caracteristiques finale.
3. La carte CAM est calculee par somme ponderee: M_c(x,y) = Somme_k (w_{k,c} * f_k(x,y)).
4. On redimensionne ensuite M_c a la taille de l'image et on l'affiche en superposition (heatmap).

### Sources citees (primaires)
- B. Zhou, A. Khosla, A. Lapedriza, A. Oliva, A. Torralba (2016), *Learning Deep Features for Discriminative Localization*, CVPR 2016. arXiv:1512.04150. https://arxiv.org/abs/1512.04150
- M. Oquab, L. Bottou, I. Laptev, J. Sivic (2015), *Is object localization for free? - Weakly-supervised learning with convolutional neural networks*, CVPR 2015. DOI: 10.1109/CVPR.2015.7298784
- K. Simonyan, A. Vedaldi, A. Zisserman (2014), *Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps*, ICLR Workshop. arXiv:1312.6034


In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def compute_cam_map(model, x_tensor, class_idx=None):
    model.eval()
    logits, fm = model(x_tensor, return_feature_maps=True)
    probs = F.softmax(logits, dim=1)

    if class_idx is None:
        class_idx = int(logits.argmax(1).item())

    weights = model.classifier.weight[class_idx]        # [K]
    fmap = fm[0]                                         # [K, H, W]
    cam = (weights[:, None, None] * fmap).sum(dim=0)    # [H, W]
    cam = torch.relu(cam)
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    return cam.cpu().numpy(), logits, probs, class_idx

def denorm_to_numpy(x_chw, mean, std):
    mean_t = torch.tensor(mean)[:, None, None]
    std_t = torch.tensor(std)[:, None, None]
    img = (x_chw.cpu() * std_t + mean_t).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

def find_ambiguous_samples(model, loader, labels, wanted_a='street', wanted_b='buildings', max_items=4):
    model.eval()
    idx_a = labels.index(wanted_a) if wanted_a in labels else None
    idx_b = labels.index(wanted_b) if wanted_b in labels else None
    selected = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = F.softmax(logits, dim=1)
            top2 = torch.topk(probs, k=2, dim=1).indices

            for i in range(xb.size(0)):
                cond_top2 = False
                if idx_a is not None and idx_b is not None:
                    t2 = set(top2[i].tolist())
                    cond_top2 = (idx_a in t2) and (idx_b in t2)

                pred_i = int(probs[i].argmax().item())
                true_i = int(yb[i].item())
                cond_label = (true_i in [idx_a, idx_b]) or (pred_i in [idx_a, idx_b])

                if cond_top2 or cond_label:
                    selected.append((xb[i].detach().cpu(), true_i))
                    if len(selected) >= max_items:
                        return selected
    return selected

echantillons_cam = find_ambiguous_samples(modele_cam, donnees_test, labels, wanted_a='street', wanted_b='buildings', max_items=4)
if len(echantillons_cam) == 0:
    print('[Q10-CAM] Aucun echantillon ambigu Street/Building detecte automatiquement; fallback sur premiers echantillons du test.')
    xb0, yb0 = next(iter(donnees_test))
    echantillons_cam = [(xb0[i], int(yb0[i].item())) for i in range(min(4, xb0.size(0)))]

fig, axes = plt.subplots(len(echantillons_cam), 3, figsize=(12, 4 * len(echantillons_cam)))
if len(echantillons_cam) == 1:
    axes = np.expand_dims(axes, axis=0)

for r, (x_cpu, y_true_i) in enumerate(echantillons_cam):
    x_in = x_cpu.unsqueeze(0).to(device)
    cam_map, logits, probs, cls_idx = compute_cam_map(modele_cam, x_in, class_idx=None)
    pred_idx = int(logits.argmax(1).item())

    img_np = denorm_to_numpy(x_cpu, NORM_MEAN, NORM_STD)

    axes[r, 0].imshow(img_np)
    axes[r, 0].set_title(f'Image | vrai={labels[y_true_i]}')
    axes[r, 0].axis('off')

    axes[r, 1].imshow(cam_map, cmap='jet')
    axes[r, 1].set_title(f'CAM brute | pred={labels[pred_idx]}')
    axes[r, 1].axis('off')

    axes[r, 2].imshow(img_np)
    axes[r, 2].imshow(cam_map, cmap='jet', alpha=0.45)
    p = float(probs[0, pred_idx].item())
    axes[r, 2].set_title(f'Superposition CAM | {labels[pred_idx]} ({p*100:.1f}%)')
    axes[r, 2].axis('off')

plt.tight_layout()
plt.show()


### Conclusion - Question 10 (CAM)
Le CAM met en evidence les zones de l'image qui soutiennent la decision du classificateur.
Sur les cas ambigus street / uildings, les cartes permettent de verifier si le modele s'appuie surtout sur les facades, la voirie, la perspective urbaine ou des textures de fond.
Cette analyse apporte une couche d'interpretabilite utile, complementaire aux metriques globales (accuracy/loss).
